# Recommendation Engine

## Objective

The objective of this notebook is to build a recommendation engine using the engineered customer features and customer segmentation results generated in previous stages of the machine learning pipeline.

Unlike previous notebooks that directly accessed MySQL, this notebook consumes the feature datasets produced by Feature Engineering and Customer Segmentation.

The recommendation engine will:

- Load engineered customer features.
- Load customer segment assignments.
- Generate recommendation candidates.
- Build popularity-based recommendations.
- Build collaborative filtering recommendations.
- Produce Top-N personalized recommendations.

The generated recommendation artifacts will later be used by the Personalization Engine and exposed through FastAPI APIs.

In [ ]:
import os 
print(os.getcwd())

In [ ]:
import os
import sys

# Find the project root notebooks/
project_root = os.path.abspath("..")

# Add it to Python's import path if not already present
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

In [ ]:
## Import Required Libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity

import joblib

from src.database import get_engine

pd.set_option("display.max_columns", None)

## Load Engineered Datasets

Load the customer feature dataset and customer segmentation dataset generated in previous notebooks.

These datasets serve as the primary inputs for the recommendation engine.

In [ ]:
#go .. one dir up = root then go in data dir and then features dir
user_item_df = pd.read_csv(os.path.join("..","data","features",
    "user_item_features.csv")
)

print(f"Dataset Shape : {user_item_df.shape}")



In [ ]:
print("=" * 50)
print("user_item_df")
print("=" * 50)

print(user_item_df.shape)

user_item_df.head()

In [ ]:
user_item_df.info()

In [ ]:
customer_segments.isnull().sum()

In [ ]:
user_item_df.describe()

In [ ]:
user_item_df.duplicated().sum()

### Observation

The user-item interaction dataset is clean and ready for recommendation modeling.

Each record represents a unique interaction between a customer and a product, with interaction strength serving as an implicit feedback score.

# Phase 1 — Popularity-Based Recommendation

Popularity-Based Recommendation is the simplest recommendation strategy.

Instead of generating personalized recommendations, it recommends products that have received the highest overall interaction from all users.

This approach serves as a baseline model and provides recommendations even for new users with no interaction history (cold-start users).

## Compute Product Popularity

Aggregate interaction strength for each product.

Products with higher cumulative interaction strength are considered more popular.